# 20 · Weight quantization: INT8, FP8, AWQ and GGUF
Level L4 · Part 3 · core · ~25 min · builds on 19

## Goal
By the end of this lesson you can pick a weight format for a memory target and measure what it saves. You can say which format fits the 70B model on the fewest L4 GPUs, and show the size of each file.

## Why it matters
- **Without it:** the 70B model in FP16 has 131 GiB of weights (lesson 19). You need eight 24 GB L4s just to hold one copy, and every extra GPU adds an all-reduce to every layer.
- **With it:** each weight takes 1 byte (INT8, FP8) or about half a byte (INT4) instead of 2. The same model fits on 4 GPUs at 8 bits or 2 GPUs at 4 bits. The change is one recipe run on the saved model, not a rewrite of the model.
- **When you don't need it:** if the model fits with room for the cache in its native format, quantizing buys little and costs quality. Lesson 21 measures the quality cost for the KV cache; for weights, build your own task check.

## The idea
Quantization stores each weight with fewer bits. A group of weights shares one **scale** number, and each weight is rounded to the nearest level the bits allow. It is like rounding prices to the nearest 5 cents and keeping one note that says how big a 5-cent step is. INT8 and FP8 use one byte per weight, INT4 about half a byte. The INT4 scheme here uses groups of 128 weights, each with one 16-bit scale, so the scales add about 0.016 bytes per weight (2 bytes ÷ 128). Some methods need no sample text at all: they round each weight from the model file alone (**data-free**). AWQ and GPTQ choose the rounding with **calibration** data, a set of sample texts run through the model first. GGUF is the single-file format of llama.cpp, used for CPU and laptop runs.

## Picture
One weight goes through a scale and a rounding step, and is multiplied back when the model runs:
```mermaid
flowchart LR
  W["FP16 weights in a group"] --> S["one scale per group"]
  S --> R["round each weight to a small integer"]
  R --> F["stored: small integers + scales"]
  F --> M["at run time: multiply back (dequantize)"]
```

## Step by step
### Step 1: a made-up model small enough for a laptop
The lesson needs a model it can save and load in a few seconds. No download is used. This cell writes a **made-up** Llama-shaped model with random weights (seed 0, about 62 million parameters, 6 layers). The file sizes are real. The answers about quality are only for this random model, not for a trained one.

In [1]:
import collections, contextlib, io
from pathlib import Path

import torch
from safetensors import safe_open
from transformers import LlamaConfig, LlamaForCausalLM

torch.manual_seed(0)                       # same numbers on every run
OUT = Path("../data/20")
OUT.mkdir(parents=True, exist_ok=True)
GIB = 2**30

cfg = LlamaConfig(hidden_size=768, intermediate_size=2048, num_hidden_layers=6,
                  num_attention_heads=12, num_key_value_heads=4, vocab_size=16000,
                  max_position_embeddings=512, tie_word_embeddings=False)
base = LlamaForCausalLM(cfg).to(torch.float16)
base.save_pretrained(OUT / "fp16", safe_serialization=True)
print(f"made-up model: {sum(p.numel() for p in base.parameters()) / 1e6:.1f}M parameters, saved as FP16")

./.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]

made-up model: 62.3M parameters, saved as FP16


**What you should see:** one line with the parameter count (about 62.3M). The file is in `../data/20/fp16`.

Peek at the first tensors. Every linear layer has a name ending in `_proj`. The embedding and the output head (`lm_head`) are not linear layers.

In [2]:
for name, p in list(base.named_parameters())[:3]:
    print(f"{name:36} {tuple(p.shape)} {p.dtype}")
print("...", len(list(base.named_parameters())), "tensors in total")

model.embed_tokens.weight            (16000, 768) torch.float16
model.layers.0.self_attn.q_proj.weight (768, 768) torch.float16
model.layers.0.self_attn.k_proj.weight (256, 768) torch.float16
... 57 tensors in total


### Step 2: the arithmetic, before any run
Bytes per weight: FP16 takes 2, INT8 and FP8 take 1, and INT4 takes 0.5 in the ideal case. The 70B model has 70.55 billion parameters (lesson 19). One L4 gives the engine 20.56 GiB. The first check is the one-GPU fit, the same check as lesson 19.

In [3]:
BYTES = {"FP16": 2, "INT8 / FP8": 1, "INT4": 0.5}   # ideal bytes per weight
L4_USABLE = 24e9 * 0.92 / GIB                       # GiB the engine gets on one L4
PARAMS_70B = 70_553_706_496                         # parameters of Llama 3.1 70B (lesson 19)
for fmt, b in BYTES.items():
    gib = PARAMS_70B * b / GIB
    verdict = "fits" if gib < L4_USABLE else "does not fit"
    print(f"{fmt:11} {gib:6.1f} GiB of weights vs {L4_USABLE:.1f} GiB on one L4: {verdict}")

FP16         131.4 GiB of weights vs 20.6 GiB on one L4: does not fit
INT8 / FP8    65.7 GiB of weights vs 20.6 GiB on one L4: does not fit
INT4          32.9 GiB of weights vs 20.6 GiB on one L4: does not fit


**What you should see:** only the ideal arithmetic is shown. Even INT4 (32.9 GiB) does not fit one L4. The real files will be a little bigger than these ideal numbers, and step 4 measures how much.

### Step 3: quantize, data-free
Three schemes, all run with no sample text. **Activations** are the numbers flowing between layers during a run. `W8A16` stores INT8 weights and keeps activations at 16 bits. `FP8_DYNAMIC` stores FP8 weights and quantizes activations to FP8 at run time. `W4A16` stores INT4 weights and keeps activations at 16 bits. `targets="Linear"` with `ignore=["lm_head"]` matches only linear layers and skips the output head, so it stays FP16. The embedding is not a linear layer either, so it is skipped and stays FP16 too.

The `WARNING ... No accelerator available` line you may see is expected on a CPU laptop. The run uses the CPU and still produces the same files.

In [4]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

SCHEMES = {"int8": "W8A16", "fp8": "FP8_DYNAMIC", "int4": "W4A16"}
for name, scheme in SCHEMES.items():
    with contextlib.redirect_stderr(io.StringIO()):   # hides progress bars and logs
        model = LlamaForCausalLM.from_pretrained(OUT / "fp16", dtype=torch.float16)
        recipe = QuantizationModifier(targets="Linear", scheme=scheme, ignore=["lm_head"])
        oneshot(model=model, recipe=recipe)           # no calibration data is passed
        model.save_pretrained(OUT / name)
    print(f"{name:5} {scheme:13} saved to ../data/20/{name}")

2026-10-24T08:22:04.5952 | get_main_device | WARNING - No accelerator available! Compressing model on CPU instead


2026-10-24T08:22:04.7940 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


2026-10-24T08:22:04.8143 | reset | INFO - Compression lifecycle reset


2026-10-24T08:22:04.8401 | from_modifiers | INFO - Creating recipe from modifiers


2026-10-24T08:22:04.8438 | get_main_device | WARNING - No accelerator available! Compressing model on CPU instead


...

2026-10-24T08:22:06.9577 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`


2026-10-24T08:22:06.9584 | get_main_device | WARNING - No accelerator available! Compressing model on CPU instead


2026-10-24T08:22:07.0270 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


2026-10-24T08:22:07.7208 | resave_config | INFO - Resaved original config with patched fields to ../data/20/int4/config.json


int4  W4A16         saved to ../data/20/int4


**What you should see:** three lines, one for each saved checkpoint. The whole lesson runs in well under a minute on a laptop CPU.

### Step 4: what each format saves on disk
The cell reads each file tensor by tensor and adds up the bytes. It reports the size of the whole file and the bytes per linear weight, which includes the scales.

In [5]:
def read_sizes(folder):
    """Bytes of the whole file, and bytes of its linear (*_proj) tensors."""
    total = linear = 0
    with safe_open(folder / "model.safetensors", "pt") as f:
        for key in f.keys():
            t = f.get_tensor(key)
            size = t.numel() * t.element_size()
            total += size
            linear += size if "_proj" in key else 0
    return total, linear

LINEAR_PARAMS = sum(p.numel() for n, p in base.named_parameters() if "_proj" in n)
FP16_TOTAL, _ = read_sizes(OUT / "fp16")
MEASURED = {}                                       # bytes per linear weight, per format
print(f"{'format':7}{'file MiB':>9}{'vs FP16':>9}{'bytes/linear weight':>21}")
for name in ["fp16", "int8", "fp8", "int4"]:
    total, linear = read_sizes(OUT / name)
    MEASURED[name] = linear / LINEAR_PARAMS
    print(f"{name:7}{total / 2**20:9.1f}{total / FP16_TOTAL:9.2f}{MEASURED[name]:21.3f}")

format  file MiB  vs FP16  bytes/linear weight
fp16       118.9     1.00                2.000
int8        83.5     0.70                1.016
fp8         83.0     0.70                1.002
int4        65.5     0.55                0.516


**What you should see:** the FP16 file is 1.00 of itself. INT8 and FP8 files are 0.70 of it, not 0.50, because the embedding and the output head stay FP16. INT4 is the smallest at 0.55, still more than a quarter of the file. The saving depends on how much of the model is linear layers, not on the bit width alone.

### Step 5: the 70B model on the L4 fleet
Now apply the **measured** bytes per linear weight to the 70B model. Its embedding and output head (2 × 128,256 × 8,192 numbers) stay FP16. For each format the cell finds the smallest tensor-parallel degree whose share of the weights fits on one L4 (degrees 1, 2, 4 and 8, as in lesson 19).

In [6]:
EMBED_70B = 2 * 128_256 * 8_192                     # input embedding + output head, FP16
WEIGHTS_GIB = {}
for name, bpp in MEASURED.items():
    gib = ((PARAMS_70B - EMBED_70B) * bpp + EMBED_70B * 2) / GIB
    WEIGHTS_GIB[name] = gib
    tp = next((t for t in (1, 2, 4, 8) if gib / t < L4_USABLE), None)
    room = "" if tp is None else f", room {L4_USABLE - gib / tp:.1f} GiB per GPU"
    print(f"{name:5} {gib:6.1f} GiB of weights -> smallest TP = {tp}{room}")

fp16   131.4 GiB of weights -> smallest TP = 8, room 4.1 GiB per GPU
int8    68.7 GiB of weights -> smallest TP = 4, room 3.4 GiB per GPU
fp8     67.8 GiB of weights -> smallest TP = 4, room 3.6 GiB per GPU
int4    36.8 GiB of weights -> smallest TP = 2, room 2.2 GiB per GPU


**What you should see:** FP16 needs TP=8 (lesson 19), and the other formats need fewer GPUs. The numbers are a little higher than the ideal ones from step 2, because of the scales and the FP16 embedding. The per-GPU room is what is left for the cache, so check it against your own memory budget.

### Step 6: does the answer still look right?
Size is only half of the decision. This smoke test runs one fixed token sequence through each model and compares the output scores (logits) with the FP16 model. It reports the relative error and how often the most likely next token is the same. Eight positions are far too few for a verdict. Lesson 21 measures quality for the KV cache with a 50-prompt set; for weights, build your own task check.

In [7]:
ids = torch.tensor([[1, 5, 77, 900, 4000, 12, 99, 3]])
with torch.no_grad():
    ref = LlamaForCausalLM.from_pretrained(OUT / "fp16", dtype=torch.float16)(ids).logits.float()
    for name in SCHEMES:
        m = LlamaForCausalLM.from_pretrained(OUT / name, dtype=torch.float16)
        out = m(ids).logits.float()
        rel = ((out - ref).norm() / ref.norm()).item()
        same = (out.argmax(-1) == ref.argmax(-1)).float().mean().item()
        print(f"{name:5} relative logit error {rel:.3f}, same top token at {round(same * 8)} of 8 positions")

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 3811.85it/s]

Applying quantization config:   0%|          | 0/42 [00:00<?, ?it/s]

Applying quantization config: 100%|██████████| 42/42 [00:00<00:00, 4894.17it/s]

Compressing model:   0%|          | 0/42 [00:00<?, ?it/s]

...

Decompressing model:  29%|██▊       | 12/42 [00:00<00:00, 114.39it/s]

Decompressing model:  57%|█████▋    | 24/42 [00:00<00:00, 114.81it/s]

Decompressing model:  86%|████████▌ | 36/42 [00:00<00:00, 111.13it/s]

Decompressing model: 100%|██████████| 42/42 [00:00<00:00, 108.60it/s]

int4  relative logit error 0.323, same top token at 3 of 8 positions


**What you should see:** the INT8 and FP8 lines read `int8  relative logit error 0.021, same top token at 8 of 8 positions` and `fp8   relative logit error 0.105, same top token at 8 of 8 positions`. Both keep the top token at all 8 positions. INT4 (round-to-nearest, no calibration) moves the scores more (0.32) and changes 5 of the 8 top tokens. A trained model can behave differently, so this only shows the method. Measure on your own model and tasks before you ship.

### Formats this lesson did not run
| Format | What it is | What it needs | Run here? |
|---|---|---|---|
| AWQ | 4-bit weights, with rounding that protects the channels most used by activations | sample text (calibration) | no |
| GPTQ | 4-bit weights, rounded with second-order information from sample text | sample text (llm-compressor's example: 512 UltraChat samples) | no |
| GGUF | one file for llama.cpp and Ollama, with K-quant types such as Q4_K_M | the llama.cpp convert and quantize tools | no (see lesson 15 for llama.cpp as an engine) |

The GPTQ recipe in llm-compressor is one line more than the FP8 one. It is shown here, not run:
```python
recipe = GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
```

**Serving the saved checkpoint (GPU, not run here).** vLLM reads the quantization settings from the checkpoint, so start it with `vllm serve ../data/20/fp8`. No GPU run was recorded, so this lesson reports no serving speed. Lesson 27, later in the course, measures it.

## At work
- **vLLM issues (keyword counts, an upper bound):** about 3,200 issues mention FP8 and about 1,250 mention AWQ, so format questions are among the most common support threads for this engine. → lesson 21 (quantizing the cache with the same tool).
- **llm-compressor** is the vLLM project's tool for FP8, INT8 and INT4 checkpoints (version 0.14.0 on PyPI). AutoAWQ's last release was in May 2025, so the maintained paths are llm-compressor and GPTQModel. → lesson 21.
- **Hardware decides the format.** FP8 tensor cores exist on Ada (L4) and Hopper (H100, H200), not on A100. A format that the card cannot compute in fast may save memory but not time. → lesson 24.

Sources: vLLM GitHub issues, https://github.com/vllm-project/vllm/issues (keyword counts recorded 2026-10-21) · llm-compressor, https://github.com/vllm-project/llm-compressor and https://pypi.org/project/llmcompressor/ · AutoAWQ on PyPI, https://pypi.org/project/autoawq/

## Common mistakes
**1. Casting a quantized model to float.** The saved model keeps its own dtypes. Calling `.float()` on it fails.

In [8]:
LlamaForCausalLM.from_pretrained(OUT / "int4", dtype=torch.float16).float()

Applying quantization config:   0%|          | 0/42 [00:00<?, ?it/s]

Applying quantization config: 100%|██████████| 42/42 [00:00<00:00, 4327.00it/s]

Compressing model:   0%|          | 0/42 [00:00<?, ?it/s]

Compressing model: 100%|██████████| 42/42 [00:00<00:00, 6666.70it/s]

Loading weights:   0%|          | 0/141 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 141/141 [00:00<00:00, 3024.96it/s]

ValueError: `.float()` is not supported for quantized model. Please use the model as it is, since the model has already been casted to the correct `dtype`.

The fix: load it with `dtype=torch.float16` and leave it as it is. Cast only the outputs, as step 6 does with `.logits.float()`.

**2. Expecting a 4× smaller file.** A 4-bit scheme makes the linear layers about 4× smaller. The FP16 embedding and output head do not shrink, so step 4 shows a much smaller saving on the whole file.

## Try it
Brightside wants the 70B model on TP=4, with at least 1 GiB of room per GPU for the cache. Which formats meet that, using `WEIGHTS_GIB` from step 5? Write the check in the cell below.

In [9]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [10]:
for name, gib in WEIGHTS_GIB.items():
    room = L4_USABLE - gib / 4
    verdict = "meets the target" if room >= 1 else "misses the target"
    print(f"{name:5} TP=4: {gib / 4:5.2f} GiB per GPU, room {room:5.2f} GiB -> {verdict}")

fp16  TP=4: 32.85 GiB per GPU, room -12.29 GiB -> misses the target
int8  TP=4: 17.17 GiB per GPU, room  3.40 GiB -> meets the target
fp8   TP=4: 16.95 GiB per GPU, room  3.61 GiB -> meets the target
int4  TP=4:  9.20 GiB per GPU, room 11.37 GiB -> meets the target


## Recap
- Quantization stores each weight in fewer bits, with one scale per group. Data-free schemes need only the model file. AWQ and GPTQ need calibration data.
- The disk saving depends on the share of linear layers, since the FP16 embedding and output head stay. Measure the whole file.
- Applied to the 70B model, the measured formats set the tensor-parallel degree (lesson 19), and the check for room per GPU sets the cache budget.
- Quantization changes outputs. Check quality on your own tasks before you ship. Lesson 21 shows the method for the KV cache.

**Interview:** "A 70B model must run on 24 GB L4s. Which weight format?" Strong answer: FP16 needs TP=8. FP8 or INT8 (about 68 GiB of weights instead of 131 GiB) fits TP=4 with room for the cache. INT4 fits TP=2, but a calibration-free 4-bit scheme needs a quality check first. I measure the saved files, not ideal arithmetic.